# Community Notes Data Inspection, Cleaning, and Validation

This notebook demonstrates inspection, cleaning outputs, and validation for four X Community Notes datasets: Note Requests, Notes, Ratings, and Note Status. The production transformations are implemented in `scripts/clean_community_notes.py`. Here, we inspect small raw samples and existing reports and Parquet outputs to avoid accidentally rescanning the very large Ratings files.

Source columns and rows are preserved, with typed derivatives and data-quality flags added. All metrics are descriptive observations; they do not establish causality or representativeness of the full Community Notes population.

In [1]:
from pathlib import Path
import json
import pandas as pd
import duckdb
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "misinformation-analysis"]
ROOT = next((p.resolve() for p in candidates if (p / "scripts" / "clean_community_notes.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate the misinformation-analysis project root.")
SOURCE_DIR = ROOT / "Project_data_1004"
OUTPUT_DIR = ROOT / "cleaned_v2"
REPORT_DIR = OUTPUT_DIR
print(f"Project: {ROOT}")
print(f"Raw sources: {SOURCE_DIR.exists()} | Cleaned outputs: {OUTPUT_DIR.exists()}")

Project: C:\Users\julia\dso576\misinformation-analysis
Raw sources: True | Cleaned outputs: True


## 1. Source Inventory and Sample

List the source shards and their file sizes, then preview at most five rows from each dataset. These previews are samples, not full-file statistics.

In [2]:
patterns = {"note_request": "batSignals-*.tsv", "notes": "notes-*.tsv", "ratings": "ratings-*.tsv", "note_status": "noteStatusHistory-*.tsv"}
files = [{"dataset": name, "file": p.name, "size_gib": p.stat().st_size / 1024**3} for name, pattern in patterns.items() for p in sorted(SOURCE_DIR.glob(pattern))]
display(pd.DataFrame(files))
sample_paths = {name: sorted(SOURCE_DIR.glob(pattern))[0] for name, pattern in patterns.items()}
raw_samples = {name: pd.read_csv(path, sep="\t", dtype=str, nrows=5) for name, path in sample_paths.items()}
for name, sample in raw_samples.items():
    print(f"{name}: {sample.shape[0]} sampled rows, {sample.shape[1]} columns")
    display(sample.head(3))

,dataset,file,size_gib
0,note_request,batSignals-00000.tsv,0.107628
1,notes,notes-00000.tsv,1.434945
2,notes,notes-00001.tsv,0.007261
3,notes,notes-00002.tsv,0.007158
4,notes,notes-00003.tsv,0.007119
5,notes,notes-00004.tsv,0.007073
6,ratings,ratings-00000.tsv,5.228990
7,ratings,ratings-00001.tsv,5.240958
8,ratings,ratings-00002.tsv,5.301669
9,ratings,ratings-00003.tsv,5.295509


note_request: 5 sampled rows, 7 columns


,tweetId,noteRequestFeedEligibleAtMillis,apiSmallFeedEligibleAtMillis,apiLargeFeedEligibleAtMillis,apiXlFeedEligibleAtMillis,sourceLinks,suggestions
0,1316941803,-1,-1,1785174077386,1773211963793,NaN,[]
1,1463732880,-1,-1,-1,1787257421863,NaN,[]
2,1711529307,-1,-1,-1,1788047004994,NaN,[]


notes: 5 sampled rows, 24 columns


,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,isCollaborativeNote
0,1783179305159200982,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978050878,1783159712986382830,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,The House failed to pass a border protection l...,0,0
1,1783181538789605871,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978583415,1783171851818021181,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,1,...,0,0,0,0,0,0,1,The United States has 50 States https://da...,0,0
2,1783182562279494134,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978827435,1783154445682979015,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,TikTok only mentions “ban” and chooses to igno...,0,0


ratings: 5 sampled rows, 35 columns


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,notHelpfulArgumentativeOrBiased,notHelpfulOffTopic,notHelpfulSpamHarassmentOrAbuse,notHelpfulIrrelevantSources,notHelpfulOpinionSpeculation,notHelpfulNoteNotNeeded,ratedOnTweetId,ratingSourceBucketed,suggestion,suggestionId
0,1352796878438424576,6BC15A6669481F9CEF45602263C22A860662F750EF683F...,1611860252442,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
1,1352796878438424576,FEB28D48AB0ACC2E768DBD320BA0E393FE7C657D904407...,1611673955881,1,1,0,0,1,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
2,1352796878438424576,A8CE8EFCAFC6F6228EEFC20E211071E6DCB40B97A050B6...,1611623203338,1,0,0,0,1,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1


note_status: 5 sampled rows, 24 columns


,noteId,noteAuthorParticipantId,createdAtMillis,timestampMillisOfFirstNonNMRStatus,firstNonNMRStatus,timestampMillisOfCurrentStatus,currentStatus,timestampMillisOfLatestNonNMRStatus,mostRecentNonNMRStatus,timestampMillisOfStatusLock,...,currentGroupStatus,currentDecidedBy,currentModelingGroup,timestampMillisOfMostRecentStatusChange,timestampMillisOfNmrDueToMinStableCrhTime,currentMultiGroupStatus,currentModelingMultiGroup,timestampMinuteOfFinalScoringOutput,timestampMillisOfFirstNmrDueToMinStableCrhTime,timestampMillisAbovePcrhThreshold
0,1352796878438424576,B021517F4F8F9F372DCD70BB8F8A21A80C12D7EFDE8B2F...,1611366884227,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,NaN,NaN,NaN,29848556,NaN,NaN
1,1353415873227177985,D4C9EDB464DF2D4A6D8F68CA8D1F86226B50EB9D3B0F33...,1611514464087,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,CoreModel (v1.1),13,-1,NaN,NaN,NaN,29848556,NaN,NaN
2,1354586938863443971,C0AF45F4C4B2240E7AB31456957E5D770FF7AFA13627C7...,1611793667892,NaN,NaN,1781032339438,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NaN,CoreModel (v1.1),NaN,-1,-1,NaN,NaN,29848556,NaN,NaN


## 2. Missingness, Categories, and Timestamps

Load the complete profiles produced by the cleaning pipeline. Missing values are not automatically imputed or dropped, and the original timestamp strings are retained. The known `-1` sentinel rule applies only to eligibility fields; it is not assumed for other timestamps.

In [3]:
manifest = pd.read_csv(REPORT_DIR / "source_manifest.csv")
missing = pd.read_csv(REPORT_DIR / "missing_values.csv")
categories = pd.read_csv(REPORT_DIR / "categorical_distributions.csv")
timestamps = pd.read_csv(REPORT_DIR / "timestamp_profile.csv")
display(manifest.groupby("dataset", as_index=False).agg(files=("file_name", "count"), rows=("row_count", "sum"), schema_ok=("schema_matches_expected", "all")))
important = {"note_request": ["tweetId", "sourceLinks"], "notes": ["noteId", "tweetId", "classification"], "ratings": ["noteId", "raterParticipantId", "helpfulnessLevel"], "note_status": ["noteId", "firstNonNMRStatus", "currentStatus"]}
selected_missing = pd.concat([missing.loc[missing.dataset.eq(ds) & missing.column.isin(cols)] for ds, cols in important.items()], ignore_index=True)
display(selected_missing[["dataset", "column", "row_count", "missing_count", "missing_percent", "decision"]])
display(categories.head(15))
display(timestamps[["dataset", "column_name", "row_count", "blank_count", "minus_one_count", "other_negative_count", "invalid_count", "valid_count"]].head(12))

,dataset,files,rows,schema_ok
0,note_request,1,489511,True
1,note_status,1,3352127,True
2,notes,5,3145882,True
3,ratings,9,236026558,True


,dataset,column,row_count,missing_count,missing_percent,decision
0,note_request,tweetId,489511,0,0.000000e+00,preserve_null_no_row_drop
1,note_request,sourceLinks,489511,373729,7.634742e-01,preserve_null_no_row_drop
2,notes,noteId,3145882,0,0.000000e+00,preserve_null_no_row_drop
3,notes,tweetId,3145882,0,0.000000e+00,preserve_null_no_row_drop
4,notes,classification,3145882,2,6.357518e-07,preserve_null_no_row_drop
5,ratings,noteId,236026558,0,0.000000e+00,preserve_null_no_row_drop
6,ratings,raterParticipantId,236026558,0,0.000000e+00,preserve_null_no_row_drop
7,ratings,helpfulnessLevel,236026558,34394,1.457209e-04,preserve_null_no_row_drop
8,note_status,noteId,3352127,0,0.000000e+00,preserve_null_no_row_drop
9,note_status,firstNonNMRStatus,3352127,2774903,8.278037e-01,preserve_null_no_row_drop


,column_name,category_value,row_count,dataset
0,classification,NOT_MISLEADING,616506,notes_clean
1,classification,MISINFORMED_OR_POTENTIALLY_MISLEADING,2529374,notes_clean
2,classification,NaN,2,notes_clean
3,helpfulnessLevel,HELPFUL,144938198,ratings_clean
4,helpfulnessLevel,NaN,34394,ratings_clean
5,helpfulnessLevel,SOMEWHAT_HELPFUL,6098171,ratings_clean
6,helpfulnessLevel,NOT_HELPFUL,84955795,ratings_clean
7,version,2,235992164,ratings_clean
8,version,1,34394,ratings_clean
9,ratingSourceBucketed,DEFAULT,233752193,ratings_clean


,dataset,column_name,row_count,blank_count,minus_one_count,other_negative_count,invalid_count,valid_count
0,note_request_clean,noteRequestFeedEligibleAtMillis,489511,0,460996,0,0,28515
1,note_request_clean,apiSmallFeedEligibleAtMillis,489511,0,460640,0,0,28871
2,note_request_clean,apiLargeFeedEligibleAtMillis,489511,0,320328,0,0,169183
3,note_request_clean,apiXlFeedEligibleAtMillis,489511,0,311,0,0,489200
4,notes_clean,createdAtMillis,3145882,0,0,0,0,3145882
5,ratings_clean,createdAtMillis,236026558,0,0,0,0,236026558
6,note_status_clean,createdAtMillis,3352127,0,0,0,0,3352127
7,note_status_clean,timestampMillisOfFirstNonNMRStatus,3352127,2774903,0,0,0,577224
8,note_status_clean,timestampMillisOfCurrentStatus,3352127,0,0,0,0,3352127
9,note_status_clean,timestampMillisOfLatestNonNMRStatus,3352127,2774903,0,0,0,577224


## 3. Cleaned Data and Table Grain

Sample the Parquet outputs to inspect retained source fields and typed derivatives. Ratings are aggregated to Note grain before joining so that the one-to-many relationship does not multiply rows in the Notes analysis table.

In [4]:
con = duckdb.connect(database=":memory:")
request_path = (OUTPUT_DIR / "note_request_clean.parquet").as_posix()
notes_path = (OUTPUT_DIR / "notes_clean.parquet").as_posix()
rating_path = (OUTPUT_DIR / "ratings_by_note.parquet").as_posix()
request_preview = con.execute("SELECT tweetId, note_request_feed_eligible_at_ms, note_request_feed_eligible_at_state, note_request_feed_eligible_at_utc, note_request_feed_eligible_at_eligible, note_request_feed_eligible_at_invalid FROM read_parquet(?) LIMIT 5", [request_path]).df()
display(request_preview)
notes_preview = con.execute("SELECT noteId, note_id, createdAtMillis, created_at_utc, created_at_state, classification, isMediaNote, isMediaNote_bool FROM read_parquet(?) LIMIT 5", [notes_path]).df()
display(notes_preview)
display(con.execute("SELECT count(*) AS note_rows, sum(rating_count) AS rating_rows, count(DISTINCT note_id) AS distinct_note_ids FROM read_parquet(?)", [rating_path]).df())

,tweetId,note_request_feed_eligible_at_ms,note_request_feed_eligible_at_state,note_request_feed_eligible_at_utc,note_request_feed_eligible_at_eligible,note_request_feed_eligible_at_invalid
0,1316941803,-1,minus_one,NaT,False,False
1,1463732880,-1,minus_one,NaT,False,False
2,1711529307,-1,minus_one,NaT,False,False
3,2095550192,-1,minus_one,NaT,False,False
4,4176682202,-1,minus_one,NaT,False,False


,noteId,note_id,createdAtMillis,created_at_utc,created_at_state,classification,isMediaNote,isMediaNote_bool
0,1945039313038926298,1945039313038926298,1752568483968,2025-07-15 08:34:43.968,valid_positive,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,False
1,1945047292459020465,1945047292459020465,1752570386411,2025-07-15 09:06:26.411,valid_positive,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,False
2,1945082274741162272,1945082274741162272,1752578726836,2025-07-15 11:25:26.836,valid_positive,MISINFORMED_OR_POTENTIALLY_MISLEADING,1,True
3,1945088555157692615,1945088555157692615,1752580224204,2025-07-15 11:50:24.204,valid_positive,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,False
4,1945449794660634843,1945449794660634843,1752666350413,2025-07-16 11:45:50.413,valid_positive,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,False


,note_rows,rating_rows,distinct_note_ids
0,3224585,236026558.0,3224585


## 4. Keys, Joins, and Validation Gates

Candidate keys and join cardinalities are validated before analysis tables are built. Unmatched keys indicate that the snapshots do not overlap; they do not prove that corresponding records do not exist in the real world.

In [5]:
duplicates = pd.read_csv(REPORT_DIR / "duplicate_validation.csv")
joins = pd.read_csv(REPORT_DIR / "join_validation.csv")
suspicious = pd.read_csv(REPORT_DIR / "suspicious_values.csv")
transformations = pd.read_csv(REPORT_DIR / "transformation_log.csv")
summary = json.loads((REPORT_DIR / "validation_summary.json").read_text(encoding="utf-8"))
display(duplicates[["dataset", "key", "method", "row_count", "unique_key_count", "repeated_key_extra_rows"]])
display(joins[["relationship", "left_rows", "right_rows", "matched_left_keys", "unmatched_left_keys", "raw_left_join_rows", "row_multiplication_factor"]])
display(suspicious.loc[pd.to_numeric(suspicious.affected_records, errors="coerce").fillna(0).gt(0)].head(20))
display(transformations.validation.value_counts(dropna=False).rename_axis("validation").to_frame("count"))
assert summary["status"] == "pass"
assert all(summary["critical_gates"].values())
assert manifest.schema_matches_expected.all()
print("Recorded critical gates: PASS")

,dataset,key,method,row_count,unique_key_count,repeated_key_extra_rows
0,note_request_clean,tweetId,exact_key_and_64bit_full_row_hash,489511,489511,0
1,notes_clean,noteId,exact_key_and_64bit_full_row_hash,3145882,3145882,0
2,note_status_clean,noteId,exact_key_and_64bit_full_row_hash,3352127,3352127,0
3,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00000.tsv,29049336,29049336,0
4,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00001.tsv,28901306,28901306,0
5,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00002.tsv,29221535,29221535,0
6,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00003.tsv,29200078,29200078,0
7,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00004.tsv,27976498,27976498,0
8,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00005.tsv,26932751,26932751,0
9,ratings_clean,noteId+raterParticipantId,64bit_candidate_key_hash:ratings-00006.tsv,27039602,27039602,0


,relationship,left_rows,right_rows,matched_left_keys,unmatched_left_keys,raw_left_join_rows,row_multiplication_factor
0,note_request_to_notes,489511,3145882,112575,376936,548952,1.121429
1,notes_to_ratings,3145882,236026558,2827454,318428,213509176,67.869417
2,notes_to_status,3145882,3352127,2904081,241801,3145882,1.000000


,dataset,issue,column_or_key,affected_records,decision,verification
30,note_status_clean,negative timestamp other than -1,timestampMillisAbovePcrhThreshold,20009,preserve raw; derived datetime null; flag,timestamp_profile.csv state reconciliation
46,ratings_clean,ratedOnTweetId differs from matched Note tweetId,ratedOnTweetId,4946147,preserve both IDs and comparison state; do not...,SELECT count(*) FROM ratings_clean r JOIN note...
47,coverage_analysis,earliest Note predates request eligibility,noteRequestFeedEligibleAtMillis,9723,preserve coverage match; use first Note at/aft...,SELECT count(*) FROM coverage_analysis WHERE n...


,count
validation,
pass,27


Recorded critical gates: PASS


## 5. Metrics, Row Reconciliation, and Optional Full Run

The four feed eligibility fields define different coverage denominators, so their results are reported separately. Response time is calculated only for Notes with a valid first non-NMR milestone; a missing milestone does not mean a Note was never resolved. The full pipeline scans approximately 236 million Ratings rows and requires substantial disk space, so the rerun below is disabled by default.

In [6]:
metrics = pd.read_csv(REPORT_DIR / "metric_summary.csv")
coverage = pd.read_csv(REPORT_DIR / "coverage_metrics_by_feed.csv")
display(metrics)
display(coverage)
source_counts = manifest.groupby("dataset").row_count.sum().to_dict()
row_checks = {"note_request_clean": source_counts["note_request"], "notes_clean": source_counts["notes"], "ratings_clean": source_counts["ratings"], "note_status_clean": source_counts["note_status"]}
row_reconciliation = pd.DataFrame([ {"table": table, "source_rows": count, "cleaned_rows": summary["output_rows"][table], "preserved": count == summary["output_rows"][table]} for table, count in row_checks.items() ])
display(row_reconciliation)
assert row_reconciliation.preserved.all()
assert (transformations.validation == "pass").all()
print("All source-grain row counts reconcile.")

# Opt in only after confirming adequate free disk space.
RUN_FULL_CLEAN = False
RERUN_DIR = ROOT / "cleaned_notebook_run"
if RUN_FULL_CLEAN:
    import subprocess, sys
    if RERUN_DIR.exists() and any(RERUN_DIR.iterdir()):
        raise FileExistsError(f"Refusing to overwrite non-empty output: {RERUN_DIR}")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "clean_community_notes.py"), "--input-dir", str(SOURCE_DIR), "--output-dir", str(RERUN_DIR), "--threads", "4", "--memory-limit", "4GB"], cwd=ROOT, check=True)
else:
    print("Skipped full-scale cleaning; RUN_FULL_CLEAN is False.")

,metric,base_unit,cohort_count,numerator_count,denominator_count,excluded_or_unobserved_count,metric_value,definition
0,response_time,note,3145882,NaN,503080,2642802,6.610761,median hours among Notes with a valid observed...
1,rating_activity,note,3145882,213190748.0,3145882,318428,67.768196,"mean matched observed Rating rows per Note, in..."
2,observed_helpfulness,rating,236026558,144938198.0,235992164,34394,0.614165,HELPFUL / valid helpfulnessLevel rows; version...


,eligibility_field,denominator_distinct_eligible_posts,numerator_posts_with_any_observed_note,observed_uncovered_posts,observed_coverage_rate
0,noteRequestFeedEligibleAtMillis,28515,19470,9045,0.682799
1,apiSmallFeedEligibleAtMillis,28871,19757,9114,0.684320
2,apiLargeFeedEligibleAtMillis,169183,74420,94763,0.439879
3,apiXlFeedEligibleAtMillis,489200,112433,376767,0.229830


,table,source_rows,cleaned_rows,preserved
0,note_request_clean,489511,489511,True
1,notes_clean,3145882,3145882,True
2,ratings_clean,236026558,236026558,True
3,note_status_clean,3352127,3352127,True


All source-grain row counts reconcile.
Skipped full-scale cleaning; RUN_FULL_CLEAN is False.
